In [27]:
import pandas as pd
import random
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
import nltk
import numpy as np

In [43]:
# Reading and cleaning the file
df = pd.read_csv('bbc_text.csv', encoding=('ISO-8859-1'))
df['clean_text'] = df['text'].str.lower()
df['clean_text'] = df['clean_text'].str.replace(r'[-/]', ' ', regex=True)
df['clean_text'] = df['clean_text'].str.replace(r'[^\w\s]', '', regex=True)
df['clean_text'] = df['clean_text'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Choosing five random samples
random_rows = df[['clean_text', 'labels', 'text']].sample(n=5)
data = random_rows.values.tolist()
data

[['tories outlining policing plans local communities would be asked to go to the polls to elect their own area police commissioner under plans unveiled by the conservatives party leader michael howard said the new role would replace inconspicuous police authorities he said the new office would not supersede the job of a chief constable the lib dems said the plan could let extreme groups run policing while labour criticised extravagant tory promises on policing responding to the plans the chairman of the police federation of england and wales which represents rank and file officers said it was essential operational independence was retained jan berry said it is a service not a political football to be kicked around every time an election approaches these plans could result in those with extreme political views dictating what actually happens on the ground she warned outlining his crime manifesto mr howard said elected police commissioners would be more accountable than police authoritie

In [45]:
# Building the modedl based on TFIDF
vectorizer = TfidfVectorizer(stop_words='english')

def tfidf_builder(passage):
    sentences = nltk.sent_tokenize(passage)
    if not sentences:
        return None, []
    cleaned_sentences = []
    for s in sentences:
        s_clean = s.lower()
        s_clean = re.sub(r'[-/]', ' ', s_clean)
        s_clean = re.sub(r'[^\w\s]', '', s_clean)
        s_clean = re.sub(r'\s+', ' ', s_clean).strip()
        cleaned_sentences.append(s_clean)
    matrix = vectorizer.fit_transform(cleaned_sentences)
    return matrix, sentences

def score_evaluation(tfidfd_matrix):
    sentence_sums = np.array(tfidf_matrix.sum(axis=1)).flatten()
    word_counts = np.diff(tfidf_matrix.indptr)
    score_list = []
    for s_sum, count in zip(sentence_sums, word_counts):
        if count > 0:
            score_list.append(s_sum / count)
        else:
            score_list.append(0.0)
    return score_list

def summarize(scores, sentences, N):
    sentence_scores = list(zip(sentences, scores))
    sorted_sentences = sorted(sentence_scores, key=lambda x: x[1], reverse=True)
    top_sentences = [sentence for sentence, score in sorted_sentences[:N]]
    return " ".join(top_sentences)

In [49]:
# Summarizing the samples
for items in data:
    text = items[2]
    lable = items[1]
    clean_text = items[0]
    tfidf_matrix, sentenc = tfidf_builder(text)
    s_list = score_evaluation(tfidf_matrix)
    print('Category:', lable)
    print('Actual text:\n', clean_text)
    print('summary:\n', summarize(s_list, sentenc, 5))
    

Category: politics
Actual text:
 tories outlining policing plans local communities would be asked to go to the polls to elect their own area police commissioner under plans unveiled by the conservatives party leader michael howard said the new role would replace inconspicuous police authorities he said the new office would not supersede the job of a chief constable the lib dems said the plan could let extreme groups run policing while labour criticised extravagant tory promises on policing responding to the plans the chairman of the police federation of england and wales which represents rank and file officers said it was essential operational independence was retained jan berry said it is a service not a political football to be kicked around every time an election approaches these plans could result in those with extreme political views dictating what actually happens on the ground she warned outlining his crime manifesto mr howard said elected police commissioners would be more acco